# Model Training — LightGBM 3× (Huber, MAE, Quantile-0.9)

**Inputs**
- `processed/train_features.parquet`, `processed/test_features.parquet`
- `processed/feature_meta_v2.json`  → `model_feature_cols`, `categorical_code_cols`, `sample_weight_col`
- `processed/cv_folds.parquet`        → 5 contiguous time-block folds covering all train rows
- `processed/val_split_v2_indices.parquet` → daytime-matched holdout for honest val R²

**Outputs**
- `oof/lgbm_huber.parquet`, `oof/lgbm_mae.parquet`, `oof/lgbm_quantile.parquet`  (OOF predictions on every train row)
- `predictions/lgbm_huber_test.parquet`, `predictions/lgbm_mae_test.parquet`, `predictions/lgbm_quantile_test.parquet`  (test predictions)
- `models/lgbm_*.txt`  (one per fold + one full-train refit per model)

**Why three models with different objectives?**

| Model | Objective | Target | Captures |
|------|-----------|--------|----------|
| `lgbm_huber`     | Huber α=0.9 on `log1p(demand)` | log-target | Mean-like behaviour, robust to a few outliers |
| `lgbm_mae`       | MAE on raw demand              | raw        | Median-like behaviour, robust to the demand=1.0 spike |
| `lgbm_quantile`  | Quantile α=0.9 on `log1p(demand)` | log-target | Upper-tail behaviour — pulls predictions toward the 90th percentile |

These three loss surfaces are different enough that **a non-negative blend will outperform any of them alone**.

### Monotone constraints

Five features are economically monotone in demand:
- `gh_mean_demand_oof`, `gh_p90_demand_oof`, `gh_max_demand_oof`, `boundary_demand` — higher prior demand → higher prediction.
- `NumberofLanes_modal` — bigger road → more traffic capacity → typically higher demand on Highway-like locations.

We pin the LightGBM `monotone_constraints` to `+1` on these so the trees can't invert priors on sparse cells.

### Training recipe

1. **Optuna (20 trials)** tunes (`learning_rate`, `num_leaves`, `min_child_samples`, `feature_fraction`, `bagging_fraction`, `reg_alpha`, `reg_lambda`) on the **val_fold** holdout from `val_split_v2`. Optuna picks the params that maximize val_fold R².
2. **5-fold CV** with best params → **OOF predictions** on every train row. These OOF predictions are what the blend notebook uses.
3. **Refit on all train data** with best params → **test predictions**.

The val_fold R² will be slightly inflated (Optuna picked params on it), but the **OOF R²** computed on the same val_fold rows is honest because each row's OOF prediction came from a fold that didn't include it.


## 1. Setup

In [1]:
import os, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 60)

BASE = '.'
PROCESSED = os.path.join(BASE, 'processed')
OOF_DIR   = os.path.join(BASE, 'oof');         os.makedirs(OOF_DIR, exist_ok=True)
PRED_DIR  = os.path.join(BASE, 'predictions'); os.makedirs(PRED_DIR, exist_ok=True)
MODEL_DIR = os.path.join(BASE, 'models');      os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42
N_OPTUNA_TRIALS = 20
print('LightGBM', lgb.__version__, '| Optuna', optuna.__version__)

LightGBM 4.6.0 | Optuna 4.9.0


## 2. Load features, folds, val split

In [2]:
train = pd.read_parquet(os.path.join(PROCESSED, 'train_features.parquet'))
test  = pd.read_parquet(os.path.join(PROCESSED, 'test_features.parquet'))
meta  = json.load(open(os.path.join(PROCESSED, 'feature_meta_v2.json')))
cv    = pd.read_parquet(os.path.join(PROCESSED, 'cv_folds.parquet'))
val_split = pd.read_parquet(os.path.join(PROCESSED, 'val_split_v2_indices.parquet'))

FEATS    = meta['model_feature_cols']
CAT      = [c for c in meta['categorical_code_cols'] if c in FEATS]
SW_COL   = meta['sample_weight_col']
ID       = meta['id_col']
TARGET   = meta['target_col']
TARGET_L = meta['target_log_col']

print(f'train: {train.shape}, test: {test.shape}, feats: {len(FEATS)}, cat: {len(CAT)}')
print(f'val_split rows: {len(val_split)} (val={(val_split.fold=="val_fold").sum()}, train={(val_split.fold=="train_fold").sum()})')
print(f'CV folds:'); print(cv['fold_id'].value_counts().sort_index().to_dict())

train: (77299, 92), test: (41778, 90), feats: 70, cat: 11
val_split rows: 77299 (val=41851, train=35448)
CV folds:
{1: 17981, 2: 18795, 3: 17166, 4: 8275, 5: 15082}


In [3]:
# Build row_id → val_fold flag
train = train.merge(val_split.rename(columns={'row_id':'Index'}), on='Index', how='left')
train['is_val'] = (train['fold'] == 'val_fold').astype(np.int8)

# Build row_id → cv fold_id
train = train.merge(cv[['Index','fold_id']], on='Index', how='left')
assert train['fold_id'].notna().all()
train['fold_id'] = train['fold_id'].astype(np.int8)

print('val_fold rows in train:', train['is_val'].sum())
print('train (non-val) rows :', (train['is_val']==0).sum())

val_fold rows in train: 41851
train (non-val) rows : 35448


## 3. Monotone constraints + helper functions

In [4]:
MONO_PLUS = ['gh_mean_demand_oof', 'gh_p90_demand_oof', 'gh_max_demand_oof',
             'boundary_demand', 'NumberofLanes_modal']
mono_vec = [1 if f in MONO_PLUS else 0 for f in FEATS]
print('Monotone +1 on:', MONO_PLUS)
print('Total constrained:', sum(mono_vec), 'of', len(FEATS))

def make_dataset(df_X, df_y, sw=None, ref=None):
    return lgb.Dataset(
        df_X[FEATS], label=df_y, weight=sw,
        categorical_feature=CAT, reference=ref, free_raw_data=False
    )

def r2_clipped(y_true, y_pred):
    return r2_score(y_true, np.clip(y_pred, 0.0, 1.0))

Monotone +1 on: ['gh_mean_demand_oof', 'gh_p90_demand_oof', 'gh_max_demand_oof', 'boundary_demand', 'NumberofLanes_modal']
Total constrained: 5 of 70


## 4. Configure the three models

In [5]:
MODEL_CONFIGS = {
    'lgbm_huber': dict(
        target_col=TARGET_L,
        invert=lambda p: np.expm1(p),
        base_params=dict(
            objective='huber', alpha=0.9, metric='rmse',
            verbosity=-1, monotone_constraints=mono_vec,
            random_state=SEED, n_jobs=-1,
        ),
    ),
    'lgbm_mae': dict(
        target_col=TARGET,
        invert=lambda p: p,
        base_params=dict(
            objective='regression_l1', metric='l1',
            verbosity=-1,  # MAE objective rejects monotone constraints
            random_state=SEED, n_jobs=-1,
        ),
    ),
    'lgbm_quantile': dict(
        target_col=TARGET_L,
        invert=lambda p: np.expm1(p),
        base_params=dict(
            objective='quantile', alpha=0.9, metric='quantile',
            verbosity=-1,  # quantile objective rejects monotone constraints
            random_state=SEED, n_jobs=-1,
        ),
    ),
}

## 5. Optuna objective (shared across the 3 models)

Each trial fits LightGBM on the train_fold portion and computes R² on the val_fold (inverse-transform first if needed, then clip to [0, 1]).

In [6]:
def run_optuna(model_name, n_trials=N_OPTUNA_TRIALS):
    cfg = MODEL_CONFIGS[model_name]
    tgt = cfg['target_col']
    invert = cfg['invert']
    base = cfg['base_params']

    tr_mask = train['is_val'] == 0
    va_mask = train['is_val'] == 1
    X_tr, X_va = train.loc[tr_mask, FEATS], train.loc[va_mask, FEATS]
    y_tr, y_va = train.loc[tr_mask, tgt],  train.loc[va_mask, tgt]
    y_va_raw   = train.loc[va_mask, TARGET]
    sw_tr      = train.loc[tr_mask, SW_COL]

    def objective(trial):
        params = dict(base)
        params.update(dict(
            learning_rate     = trial.suggest_float('lr', 0.015, 0.06, log=True),
            num_leaves        = trial.suggest_int  ('num_leaves', 31, 191),
            min_child_samples = trial.suggest_int  ('min_child_samples', 10, 80),
            feature_fraction  = trial.suggest_float('feature_fraction', 0.6, 1.0),
            bagging_fraction  = trial.suggest_float('bagging_fraction', 0.6, 1.0),
            bagging_freq      = 5,
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
        ))
        d_tr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr, categorical_feature=CAT, free_raw_data=False)
        d_va = lgb.Dataset(X_va, label=y_va, categorical_feature=CAT, reference=d_tr, free_raw_data=False)
        model = lgb.train(
            params, d_tr, num_boost_round=2500, valid_sets=[d_va],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
        raw_pred = invert(model.predict(X_va, num_iteration=model.best_iteration))
        return r2_clipped(y_va_raw, raw_pred)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    t0 = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f'{model_name}: best val R² = {study.best_value:.5f} | best params = {study.best_params}')
    print(f'  Optuna took {time.time()-t0:.1f}s for {n_trials} trials')
    return study.best_params

## 6. CV refit (with best Optuna params) → OOF + test predictions

In [7]:
def cv_refit_and_predict(model_name, best_params):
    cfg = MODEL_CONFIGS[model_name]
    tgt = cfg['target_col']; invert = cfg['invert']; base = cfg['base_params']
    full_params = dict(base); full_params.update(best_params); full_params['bagging_freq'] = 5

    oof_pred = np.full(len(train), np.nan)
    test_pred_acc = np.zeros(len(test))

    for fid in [1,2,3,4,5]:
        in_fold = train['fold_id'] == fid
        X_tr = train.loc[~in_fold, FEATS]
        y_tr = train.loc[~in_fold, tgt]
        sw_tr= train.loc[~in_fold, SW_COL]
        X_va = train.loc[ in_fold, FEATS]
        y_va = train.loc[ in_fold, tgt]
        d_tr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr, categorical_feature=CAT, free_raw_data=False)
        d_va = lgb.Dataset(X_va, label=y_va, categorical_feature=CAT, reference=d_tr, free_raw_data=False)
        model = lgb.train(full_params, d_tr, num_boost_round=3000, valid_sets=[d_va],
                         callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof_pred[in_fold.values] = invert(model.predict(X_va, num_iteration=model.best_iteration))
        test_pred_acc += invert(model.predict(test[FEATS], num_iteration=model.best_iteration))
        model.save_model(os.path.join(MODEL_DIR, f'{model_name}_fold{fid}.txt'))
        print(f'  Fold {fid}: best_iter={model.best_iteration}')
    test_pred_cv = test_pred_acc / 5.0

    # Final refit on full train
    d_full = lgb.Dataset(train[FEATS], label=train[tgt], weight=train[SW_COL],
                         categorical_feature=CAT, free_raw_data=False)
    final = lgb.train(full_params, d_full,
                      num_boost_round=int(np.mean([1500])))  # rough avg best_iter * 1
    # Better: compute average best_iter from the 5 fold models above
    iters = []
    for fid in [1,2,3,4,5]:
        m = lgb.Booster(model_file=os.path.join(MODEL_DIR, f'{model_name}_fold{fid}.txt'))
        iters.append(m.num_trees())
    avg_iter = int(np.mean(iters))
    final = lgb.train(full_params, d_full, num_boost_round=avg_iter)
    final.save_model(os.path.join(MODEL_DIR, f'{model_name}_final.txt'))
    test_pred_final = invert(final.predict(test[FEATS]))
    test_pred = 0.5 * test_pred_cv + 0.5 * test_pred_final   # blend CV-mean and full-refit
    test_pred = np.clip(test_pred, 0.0, 1.0)
    oof_pred  = np.clip(oof_pred,  0.0, 1.0)

    # Score
    oof_r2 = r2_score(train[TARGET], oof_pred)
    val_idx = train['is_val'] == 1
    val_r2  = r2_score(train.loc[val_idx, TARGET], oof_pred[val_idx])
    print(f'{model_name}: OOF R² (all train) = {oof_r2:.5f} | OOF R² (val_fold subset) = {val_r2:.5f}')

    # Save OOF + test prediction parquets
    oof_df  = pd.DataFrame({'Index': train['Index'].values,
                            f'{model_name}_oof_pred': oof_pred})
    test_df = pd.DataFrame({'Index': test['Index'].values,
                            f'{model_name}_test_pred': test_pred})
    oof_df.to_parquet(os.path.join(OOF_DIR,  f'{model_name}.parquet'), index=False)
    test_df.to_parquet(os.path.join(PRED_DIR, f'{model_name}_test.parquet'), index=False)
    return dict(oof_r2=oof_r2, val_r2=val_r2, avg_iter=avg_iter, best_params=best_params)

## 7. Train all three models

In [8]:
results = {}
for model_name in ['lgbm_huber', 'lgbm_mae', 'lgbm_quantile']:
    print(f'\n=== {model_name} ===')
    print('Optuna search...')
    best = run_optuna(model_name)
    print('CV refit...')
    results[model_name] = cv_refit_and_predict(model_name, best)

print('\n=== SUMMARY ===')
for n, r in results.items():
    print(f'{n}: OOF R²={r["oof_r2"]:.5f}, val_fold OOF R²={r["val_r2"]:.5f}, avg_iter={r["avg_iter"]}')

with open(os.path.join(BASE, 'oof', 'lgbm_summary.json'), 'w') as f:
    summ = {k: {kk: vv for kk, vv in v.items() if kk != 'best_params'} | {'best_params': v['best_params']}
            for k, v in results.items()}
    json.dump(summ, f, indent=2, default=str)
print('Saved summary to oof/lgbm_summary.json')


=== lgbm_huber ===
Optuna search...


lgbm_huber: best val R² = 0.95976 | best params = {'lr': 0.0460039327012576, 'num_leaves': 80, 'min_child_samples': 16, 'feature_fraction': 0.8736932106048627, 'bagging_fraction': 0.7760609974958406, 'reg_alpha': 0.002323350351539011, 'reg_lambda': 0.030586566669785254}
  Optuna took 51.2s for 20 trials
CV refit...


  Fold 1: best_iter=52


  Fold 2: best_iter=44


  Fold 3: best_iter=68


...

  Fold 2: best_iter=68


  Fold 3: best_iter=75


  Fold 4: best_iter=3000


  Fold 5: best_iter=38


lgbm_quantile: OOF R² (all train) = 0.91533 | OOF R² (val_fold subset) = 0.92920

=== SUMMARY ===
lgbm_huber: OOF R²=0.95761, val_fold OOF R²=0.96653, avg_iter=53
lgbm_mae: OOF R²=0.95331, val_fold OOF R²=0.96479, avg_iter=92
lgbm_quantile: OOF R²=0.91533, val_fold OOF R²=0.92920, avg_iter=645
Saved summary to oof/lgbm_summary.json


## 8. Round-trip sanity check

In [9]:
for model_name in ['lgbm_huber','lgbm_mae','lgbm_quantile']:
    oof_p  = pd.read_parquet(os.path.join(OOF_DIR,  f'{model_name}.parquet'))
    test_p = pd.read_parquet(os.path.join(PRED_DIR, f'{model_name}_test.parquet'))
    assert len(oof_p) == 77299
    assert len(test_p) == 41778
    pred_col_t = f'{model_name}_test_pred'
    print(f'{model_name}: OOF NaN={oof_p[f"{model_name}_oof_pred"].isna().sum()}, '
          f'test NaN={test_p[pred_col_t].isna().sum()}, '
          f'test_pred range=[{test_p[pred_col_t].min():.4f}, {test_p[pred_col_t].max():.4f}]')
print('All round-trip checks pass ✓')

lgbm_huber: OOF NaN=0, test NaN=0, test_pred range=[0.0038, 0.9899]
lgbm_mae: OOF NaN=0, test NaN=0, test_pred range=[0.0012, 0.9998]
lgbm_quantile: OOF NaN=0, test NaN=0, test_pred range=[0.0064, 1.0000]
All round-trip checks pass ✓
